# Bayesian Network Structure Learning — POC Experiment

Compares **Hill Climbing** (pgmpy) against **NOTEARS** (gcastle) in two feature regimes:
- Low-dimensional: 10 features, 1000 samples
- High-dimensional: 100 features, 1000 samples

Each dataset is split 90/10 into train/test. We measure wall-clock time for structure learning.

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from data import generate_random_dataset, build_random_dag
from training import train_hill_climbing, train_notears

In [ ]:
SEED = 42
N_SAMPLES = 1000
REGIMES = {
    "low": 4, 
    "high": 8,
}

In [ ]:
datasets = {}
for regime, n_features in REGIMES.items():
    random_dag = build_random_dag(n_nodes=n_features, seed=SEED)
    df = generate_random_dataset(n_features=n_features, n_samples=N_SAMPLES, seed=SEED, ground_truth_dag=random_dag)
    train, test = train_test_split(df, test_size=0.1, random_state=SEED)
    datasets[regime] = {"train": train.reset_index(drop=True), "test": test.reset_index(drop=True), "dag": random_dag}
    print(f"[{regime}] train={train.shape}, test={test.shape}")

In [ ]:
results = []

for regime, splits in datasets.items():
    train = splits["train"]
    gt_dag = splits["dag"]
    columns = list(train.columns)
    true_edges = set(gt_dag.edges())

    # --- Hill Climbing ---
    t0 = time.perf_counter()
    hc_model = train_hill_climbing(train)
    hc_time = time.perf_counter() - t0

    hc_edges = set(hc_model.edges())
    results.append({
        "regime": regime, "algorithm": "HillClimbing",
        "time_s": hc_time, "n_edges": len(hc_edges),
        "learned_edges": hc_edges, "true_edges": true_edges,
    })
    print(f"[{regime}] HillClimbing: {hc_time:.2f}s, edges={len(hc_edges)}")

    # --- NOTEARS ---
    t0 = time.perf_counter()
    nt_model = train_notears(train)
    nt_time = time.perf_counter() - t0

    # Reconstruct edge set from the adjacency matrix (row -> col means row causes col)
    adj = nt_model.causal_matrix
    nt_edges = {
        (columns[i], columns[j])
        for i in range(len(columns))
        for j in range(len(columns))
        if adj[i, j] != 0
    }
    results.append({
        "regime": regime, "algorithm": "NOTEARS",
        "time_s": nt_time, "n_edges": len(nt_edges),
        "learned_edges": nt_edges, "true_edges": true_edges,
    })
    print(f"[{regime}] NOTEARS:      {nt_time:.2f}s, edges={len(nt_edges)}")

In [ ]:
summary = pd.DataFrame(results)
summary["time_s"] = summary["time_s"].round(3)
summary

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, metric, label in zip(axes, ["time_s", "n_edges"], ["Time (s)", "Learned edges"]):
    pivot = summary.pivot(index="regime", columns="algorithm", values=metric)
    pivot.plot(kind="bar", ax=ax, rot=0)
    ax.set_title(label)
    ax.set_xlabel("Feature regime")
    ax.set_ylabel(label)

fig.suptitle("Hill Climbing vs NOTEARS", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## Graph recovery metrics

We compare the learned DAG against the ground-truth using standard structure-learning metrics computed on directed edges:

| Metric | Definition |
|--------|-----------|
| **SHD** | Structural Hamming Distance — number of edge insertions, deletions and reversals to turn the learned graph into the true one (lower is better) |
| **Precision** | TP / (TP + FP + reversed) — fraction of learned edges that are correct |
| **Recall** | TP / (TP + FN + reversed) — fraction of true edges that were recovered |
| **F1** | Harmonic mean of precision and recall |

Reversed edges count as a single mistake in SHD and are excluded from both TP buckets for precision/recall.

In [ ]:
def graph_metrics(true_edges: set, learned_edges: set) -> dict:
    """Compute directed-edge recovery metrics between two edge sets.

    Reversed edges (u->v learned when v->u is true) count as 1 in SHD
    and are not counted as TP for either precision or recall.
    """
    reversed_edges = {(v, u) for u, v in true_edges} & learned_edges

    tp = len(true_edges & learned_edges)
    fp = len(learned_edges - true_edges - reversed_edges)
    fn = len(true_edges - learned_edges - {(v, u) for u, v in reversed_edges})
    rev = len(reversed_edges)

    shd = fp + fn + rev
    precision = tp / (tp + fp + rev) if (tp + fp + rev) > 0 else 0.0
    recall    = tp / (tp + fn + rev) if (tp + fn + rev) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)

    return {"shd": shd, "precision": precision, "recall": recall, "f1": f1,
            "tp": tp, "fp": fp, "fn": fn, "reversed": rev}


metrics_rows = []
for r in results:
    m = graph_metrics(r["true_edges"], r["learned_edges"])
    metrics_rows.append({
        "regime": r["regime"],
        "algorithm": r["algorithm"],
        "true_edges": len(r["true_edges"]),
        "learned_edges": r["n_edges"],
        **m,
    })

metrics_df = pd.DataFrame(metrics_rows)
metrics_df[["precision", "recall", "f1"]] = metrics_df[["precision", "recall", "f1"]].round(3)
metrics_df

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
metric_cols = ["shd", "precision", "recall", "f1"]
labels      = ["SHD ↓", "Precision ↑", "Recall ↑", "F1 ↑"]

for ax, col, lbl in zip(axes, metric_cols, labels):
    pivot = metrics_df.pivot(index="regime", columns="algorithm", values=col)
    pivot.plot(kind="bar", ax=ax, rot=0)
    ax.set_title(lbl)
    ax.set_xlabel("Feature regime")
    ax.set_ylabel(lbl)

fig.suptitle("Structure recovery: Hill Climbing vs NOTEARS", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()